# DMRC RAG — 02. Gemma 2 9B Inference & API Serving

Loads the `chroma_db/` built by **`01_Setup_and_Retrieval_Validation.ipynb`**, loads the BM25 index, the cross-encoder reranker, and Gemma 2 9B, then starts the FastAPI app (`src/app.py`) and runs full end-to-end RAG queries against it.

**This notebook never regenerates embeddings or rebuilds ChromaDB.** It assumes notebook 1 already completed successfully and simply loads what it produced. Needs a **GPU runtime** (Runtime -> Change runtime type -> GPU).

### 1. Clone / update the repository

In [ ]:
%cd /content
!test -d dmrc_deploy && (echo "dmrc_deploy/ already present -- pulling latest" && cd dmrc_deploy && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy.git
%cd /content/dmrc_deploy

### 2. Install dependencies

Same core install as notebook 1, plus a best-effort install of the 4-bit extras (`bitsandbytes` / `nvidia-nvjitlink-cu13`) -- only needed if `GEMMA_USE_4BIT=1` is set later in this notebook; harmless to skip if either wheel fails to install here.

In [ ]:
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt
!pip install -q -r /tmp/requirements_core.txt

!pip install -q "bitsandbytes>=0.43.0" nvidia-nvjitlink-cu13 \
    || echo "4-bit extras failed to install -- fine if staying on the default bf16 path (GEMMA_USE_4BIT=0)."

In [ ]:
import os
os.kill(os.getpid(), 9)

### 3. Confirm GPU + CUDA

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- switch this runtime to GPU before continuing."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 4. Hugging Face login

`google/gemma-2-9b-it` (`src/gemma_inference.py`) is a **gated** checkpoint -- the account behind this login must have accepted its license at https://huggingface.co/google/gemma-2-9b-it, or the model load below fails with a 401/gated-repo error.

In [ ]:
from huggingface_hub import login
login()

### 5. Load the ChromaDB built by Notebook 1

Read-only connection to the existing `chroma_db/` -- this notebook must **not** rebuild it. If this cell fails or reports 0 vectors, go run notebook 1 first.

In [ ]:
from src.storage import get_collection, COLLECTION_NAME

assert os.path.exists("./chroma_db"), (
    "chroma_db/ not found -- run 01_Setup_and_Retrieval_Validation.ipynb first."
)

collection = get_collection()
total = collection.count()
print(f"Collection : {COLLECTION_NAME}")
print(f"Total vectors: {total}")

assert total > 0, "chroma_db/ exists but is empty -- run notebook 1 to (re)build it."
print("[OK] Existing ChromaDB collection loaded.")

### 6. Load the BM25 index

`bm25_index.get_bm25_index()` pulls every chunk's text + metadata straight out of the ChromaDB collection just confirmed above and builds the lexical index from it -- no separate BM25 storage exists.

In [ ]:
import src.bm25_index as bm25_index

bm25 = bm25_index.get_bm25_index()
print(f"BM25 index built over {len(bm25.chunk_ids)} chunks.")

### 7. Load the cross-encoder reranker

`BAAI/bge-reranker-v2-m3`, used by `reranker.rerank()` in the retrieval pipeline below.

In [ ]:
import src.reranker as reranker

reranker.get_reranker_model()
print("Reranker model loaded.")

### 8. Load Gemma 2 9B and smoke-test `generate_answer`

In [ ]:
from src.gemma_inference import get_gemma_model, generate_answer

model, tokenizer, device = get_gemma_model()
print(f"Model loaded. device={device}")

answer = generate_answer("What is Artificial Intelligence?")
print("\n" + answer)

### 9. Release this notebook's copy of the model before starting the server

The FastAPI server started below loads its own copy of every model (including Gemma) inside its own subprocess on first request / at startup (`app.py`'s `lifespan` warm-up). Freeing this notebook kernel's reference first avoids holding two full copies of Gemma 2 9B in VRAM at once.

In [ ]:
import gc

for _name in ("model", "tokenizer"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
print("Notebook-side model reference cleared; GPU memory released.")

### 10. Start the FastAPI server

`src/app.py` already imports `src/retrieval_caps.py` directly at module load time (no `%%writefile`/`sitecustomize.py` trick needed -- that file is a committed, version-controlled module now, not something the notebook has to generate). `RAG_MAX_CANDIDATES` / `RAG_MAX_CONTEXT` are deliberately **not** overridden here: the repository default (`RAG_MAX_CONTEXT=15`, see `retrieval_caps.py`) is what should actually govern retrieval breadth -- a prior Colab runner overriding `RAG_MAX_CONTEXT=4` here was the root cause of a previously-fixed "answers grounded in only 4 clauses" bug, so this notebook intentionally leaves that default alone.

In [ ]:
import subprocess, time, requests

server_env = {
    **os.environ,
    "GEMMA_USE_4BIT": "0",
    "GEMMA_MAX_NEW_TOKENS": "320",
}

server = subprocess.Popen(
    ["uvicorn", "src.app:app", "--host", "127.0.0.1", "--port", "8000"],
    env=server_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("Starting server (bf16, repository-default retrieval caps)...")
start = time.time()
while True:
    if server.poll() is not None:
        print("Server exited early:")
        print(server.stdout.read())
        break
    try:
        r = requests.get("http://127.0.0.1:8000/status", timeout=3)
        if r.status_code == 200:
            print(f"HTTP up after {time.time()-start:.0f}s -> {r.json()}")
            break
    except requests.exceptions.RequestException:
        pass
    print(f"  ...{time.time()-start:.0f}s")
    time.sleep(5)

### 11. Warm up

First `/ask` call pays the full model-load cost (`lifespan`'s warm-up already loads the dense + reranker + Gemma models, but a first real request through the whole stack is still the most reliable warm-up).

In [ ]:
print("Warming up (first call includes the full model load)...")
t0 = time.time()
try:
    w = requests.post("http://127.0.0.1:8000/ask", json={"query": "warmup"}, timeout=1800)
    print(f"Warm-up done in {time.time()-t0:.0f}s -- status {w.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Warm-up failed after {time.time()-t0:.0f}s: {e}")
print("\nModel resident. Timings below are pure inference.")

### 12. Test every API endpoint

`GET /` (health), `GET /status` (model/DB readiness), and `POST /admin/reload-bm25` (BM25 rebuild) -- everything except `/ask`, which gets its own dedicated section below.

In [ ]:
health = requests.get("http://127.0.0.1:8000/").json()
print("GET / ->", health)

status = requests.get("http://127.0.0.1:8000/status").json()
print("GET /status ->", status)
assert status["chromadb_connected"], "Server cannot reach ChromaDB."
assert status["dense_model_loaded"], "Dense embedding model not loaded."
assert status["reranker_model_loaded"], "Reranker model not loaded."
assert status["gemma_model_loaded"], "Gemma model not loaded."

reload_resp = requests.post("http://127.0.0.1:8000/admin/reload-bm25").json()
print("POST /admin/reload-bm25 ->", reload_resp)

print("\n[OK] /, /status, and /admin/reload-bm25 all responded correctly.")

### 13. Retrieved chunks vs reranked chunks (one query, both stages shown)

Calls `hybrid_search()` and `rerank()` directly (bypassing the API) for a single example query, so the raw retrieved candidates and the reranked candidates can be inspected side by side before looking at the final served `/ask` responses below.

In [ ]:
import src.hybrid_retriever as hr
import src.reranker as rr

EXAMPLE_QUERY = "Explain Fire Alarm Control Panels and corresponding BOQ items."

retrieved = hr.hybrid_search(EXAMPLE_QUERY)
print(f"Retrieved candidates ({len(retrieved)}):")
for c in retrieved[:10]:
    md_ = c.get("metadata") or {}
    print(f"  {c['chunk_id']}  chunk_type={md_.get('chunk_type')}  "
          f"source={c.get('retrieval_source')}  score={c.get('score')}")

reranked = rr.rerank(EXAMPLE_QUERY, retrieved)
print(f"\nReranked candidates ({len(reranked)}):")
for c in reranked[:10]:
    md_ = c.get("metadata") or {}
    print(f"  {c['chunk_id']}  chunk_type={md_.get('chunk_type')}  "
          f"reranker_score={c.get('reranker_score')}")

### 14. End-to-end RAG queries -- Clause, BOQ, and Hybrid

Runs every query below through the live `/ask` endpoint (retrieval -> rerank -> prompt -> Gemma generation -> JSON response) and prints the sources plus Gemma's final answer for each.

In [ ]:
CLAUSE_QUERIES = [
    "What are the requirements for fire alarm systems?",
    "Explain Clause 6.7.2.",
    "What are the testing requirements?",
    "Explain signalling requirements.",
]

BOQ_QUERIES = [
    "Find BOQ Item 1.01.",
    "Find MCC panels.",
    "Find UPS items.",
    "Show Schedule A panels.",
    "Find quantity for BOQ Item 2.03.",
]

HYBRID_QUERIES = [
    "Explain Fire Alarm Control Panels and corresponding BOQ items.",
    "Which BOQ item corresponds to signalling equipment?",
    "Show specification and BOQ entries for MCC panels.",
]


def ask(query, timeout=300):
    t0 = time.time()
    try:
        r = requests.post("http://127.0.0.1:8000/ask", json={"query": query}, timeout=timeout)
        dt = time.time() - t0
        if r.status_code == 200:
            return dt, r.json(), None
        return dt, None, f"HTTP {r.status_code}: {r.text[:200]}"
    except requests.exceptions.RequestException as e:
        return time.time() - t0, None, f"{type(e).__name__}: {e}"


def run_category(label, queries):
    print("=" * 74)
    print(label)
    print("=" * 74)
    results = []
    for q in queries:
        dt, data, error = ask(q)
        if data is not None:
            sources = data.get("sources", [])
            how = sources[0].get("retrieval_source") if sources else "-"
            print(f"\nQ: {q}\n   [{dt:.1f}s | {len(sources)} sources | path: {how} | "
                  f"confidence: {data.get('confidence')}]")
            print(f"\n{data.get('answer', '')}")
            results.append((q, dt, True))
        else:
            print(f"\nQ: {q}\n   FAILED after {dt:.0f}s: {error}")
            results.append((q, dt, False))
    return results


all_results = []
all_results += run_category("CLAUSE QUERIES", CLAUSE_QUERIES)
all_results += run_category("BOQ QUERIES", BOQ_QUERIES)
all_results += run_category("HYBRID QUERIES", HYBRID_QUERIES)

print(f"\n\n{'='*74}\nSUMMARY")
for q, dt, ok in all_results:
    print(f"  {'OK  ' if ok else 'FAIL'} {dt:6.1f}s  {q[:56]}")
ok_count = sum(1 for _, _, ok in all_results if ok)
print(f"\n  {ok_count}/{len(all_results)} succeeded")

assert ok_count == len(all_results), "One or more /ask queries failed -- see FAIL lines above."
print("\n[OK] All clause, BOQ, and hybrid RAG queries succeeded end-to-end.")

### 15. Shut down

In [ ]:
server.terminate()
try:
    server.wait(timeout=30)
    print("Server stopped.")
except subprocess.TimeoutExpired:
    server.kill()
    server.wait(timeout=10)
    print("Server did not exit gracefully in 30s; force-killed it.")